# Convert Benchmark (PFF→L0 Zarr and in-memory fast path)\n",
    "\n",
    "Measures PFF→L0 write throughput across configurations, plus the new **`skip_l0_materialize`**\n",
    "mode that skips the L0 on-disk store entirely and goes straight to L1 in memory.\n",
    "\n",
    "| Variant | What happens | When to use |\n",
    "|---------|-------------|-------------|\n",
    "| Baseline (no checksum, no sharding) | PFF → L0 Zarr | R&D convert |\n",
    "| + checksum | Adds a full second I/O re-read | Production (integrity) |\n",
    "| + sharding (`shard_factor=16`) | Fewer files → faster `xr.open_zarr` on BeeGFS | HPC/Expanse |\n",
    "| `skip_l0_materialize=True` | PFF → (in-memory L0) → L1 Zarr | Fastest L1; no L0 on disk |

In [ ]:
from pathlib import Path

from panoseti_analysis.io.bench import BenchResult, stage_timer, summarize
from panoseti_analysis.paths import REPO_ROOT

# ── Configure ───────────────────────────────────────────────────────────────────────────────
OBS_DIR = Path("/path/to/obs.pffd")  # replace with a real .pffd run
OUT_BASE = Path("/tmp/convert_bench")

results: list[BenchResult] = []

## §1 Baseline: ZarrPythonWriter, no checksum, no sharding

In [ ]:
from panoseti_analysis.adapters.convert import run_convert

OUT_DIR = OUT_BASE / "baseline"
out_bytes = 0  # fill after run

with stage_timer("convert_baseline", bytes_in=0) as r:
    records = run_convert(OBS_DIR, OUT_DIR, checksum=False)
    out_bytes = sum(f.stat().st_size for f in OUT_DIR.rglob("*") if f.is_file())

r.bytes_out = out_bytes
results.append(r)
print(summarize(results))

## §2 With checksum

In [ ]:
OUT_DIR = OUT_BASE / "with_checksum"
out_bytes = 0  # fill after run

with stage_timer("convert_with_checksum", bytes_in=0) as r:
    records = run_convert(OBS_DIR, OUT_DIR, checksum=True)
    out_bytes = sum(f.stat().st_size for f in OUT_DIR.rglob("*") if f.is_file())

r.bytes_out = out_bytes
results.append(r)
print(summarize(results))

## §3 With sharding (shard_factor=16)

In [ ]:
OUT_DIR = OUT_BASE / "sharded"
out_bytes = 0  # fill after run

with stage_timer("convert_sharded", bytes_in=0) as r:
    records = run_convert(OBS_DIR, OUT_DIR, checksum=False, shard_factor=16)
    out_bytes = sum(f.stat().st_size for f in OUT_DIR.rglob("*") if f.is_file())

r.bytes_out = out_bytes
results.append(r)
print(summarize(results))

## §4 File count comparison (baseline vs sharded)

In [ ]:
for label, out_dir in [("baseline", OUT_BASE / "baseline"), ("sharded", OUT_BASE / "sharded")]:
    files = list(out_dir.rglob("*"))
    print(f"{label}: {len(files)} files")

## §5 skip_l0_materialize — PFF → L1 with no L0 on disk

`run_pipeline(skip_l0_materialize=True)` uses `sequence_to_dataset` to build an in-memory
L0 Dataset for each product, then calls `run_calibrate_inmem` to produce L1 on disk without
ever writing L0 Zarr chunks. The L0 data lives only in RAM during calibration.

In [ ]:
from panoseti_analysis.adapters.recipe_driver import run_pipeline
from panoseti_analysis.config.recipes import load_recipe

RECIPE = REPO_ROOT / "recipes/img_calib_default.yml"
recipe_params, recipe_name, recipe_hash = load_recipe(RECIPE)

# -- §5a: full pipeline (L0 → L1 on disk) for comparison ---------------------
OUT_FULL = OUT_BASE / "full_pipeline"
with stage_timer("full_pipeline (L0+L1 on disk)", bytes_in=0) as r_full:
    _ = run_pipeline(OBS_DIR, OUT_FULL, recipe=recipe_params, skip_l0_materialize=False)
r_full.bytes_out = sum(f.stat().st_size for f in OUT_FULL.rglob("*") if f.is_file())
results.append(r_full)

# -- §5b: skip_l0_materialize (L1 only on disk) --------------------------------
OUT_SKIPL0 = OUT_BASE / "skip_l0"
with stage_timer("skip_l0_materialize (L1 only on disk)", bytes_in=0) as r_skip:
    _ = run_pipeline(OBS_DIR, OUT_SKIPL0, recipe=recipe_params, skip_l0_materialize=True)
r_skip.bytes_out = sum(f.stat().st_size for f in OUT_SKIPL0.rglob("*") if f.is_file())
results.append(r_skip)

print(summarize(results))
print(f"\nDisk saved by skipping L0: {(r_full.bytes_out - r_skip.bytes_out) / 1e9:.2f} GB")